# 1. Khởi tạo môi trường và thư viện


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


# 2. Hàm đọc dữ liệu


In [ ]:
def load_dataset(data_path: str) -> pd.DataFrame:
    df = pd.read_csv(data_path, sep='	')
    expected_cols = ['ContextID', 'Compound', 'Mod', 'Head', 'Context']
    for col in ['ModAvg', 'ModStd', 'HeadAvg', 'HeadStd']:
        if col not in df.columns and col.replace('Avg', '') in df.columns:
            df[col] = df[col.replace('Avg', '')]
    return df

# Ví dụ nạp dữ liệu (thay đường dẫn thực tế khi chạy)
# df = load_dataset('train.tsv')


# 3. Phân tích khám phá dữ liệu (EDA)


In [ ]:
# Kiểm tra tổng quan và giá trị khuyết thiếu
# df.info()
# df.head()

# Thống kê phân phối điểm Mod và Head
# print(df[['Mod', 'Head', 'ModAvg', 'HeadAvg']].describe())

# Trực quan hóa phân phối nhãn Mod và Head
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# sns.histplot(df['Mod'], bins=20, kde=True, ax=axes[0], color='skyblue')
# axes[0].set_title('Phân phối điểm Mod')
# sns.histplot(df['Head'], bins=20, kde=True, ax=axes[1], color='salmon')
# axes[1].set_title('Phân phối điểm Head')
# plt.tight_layout()
# plt.show()

# Kiểm tra tương quan giữa Mod và Head
# corr, _ = spearmanr(df['Mod'], df['Head'])
# print(f"Spearman correlation giữa Mod và Head: {corr:.4f}")

# Đo lường phương sai trong compound (Intra-compound variance)
# comp_stats = df.groupby('Compound').agg({'Mod': ['mean', 'std'], 'Head': ['mean', 'std']})
# print("Tỷ lệ compound có phương sai nhãn > 0:")
# print((comp_stats[('Mod', 'std')] > 0).mean())


# 4. Phân chia dữ liệu StratifiedGroupKFold theo Target


In [ ]:
def split_data(df: pd.DataFrame, n_splits: int = 5, seed: int = 42) -> pd.DataFrame:
    df = df.copy()
    comp_target = df.groupby('Compound')['Mod'].mean().reset_index()
    comp_target['mod_bin'] = pd.qcut(comp_target['Mod'], q=5, labels=False, duplicates='drop')
    
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    df['fold'] = -1
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(comp_target, comp_target['mod_bin'], groups=comp_target['Compound'])):
        val_compounds = comp_target.iloc[val_idx]['Compound'].values
        df.loc[df['Compound'].isin(val_compounds), 'fold'] = fold
        
    return df

# df = split_data(df, n_splits=5)
# print(df.groupby('fold')['Compound'].nunique())


# 5. Hàm đánh giá (Metrics Evaluator)


In [ ]:
def compute_metrics(y_true_mod, y_pred_mod, y_true_head, y_pred_head, compounds):
    # 1. Main metric: Overall Spearman Correlation
    spearman_mod, _ = spearmanr(y_true_mod, y_pred_mod)
    spearman_head, _ = spearmanr(y_true_head, y_pred_head)
    main_spearman = (spearman_mod + spearman_head) / 2.0
    
    # 2. Sub metric: Context Variance Correlation
    df_eval = pd.DataFrame({
        'comp': compounds,
        'true_mod': y_true_mod,
        'pred_mod': y_pred_mod,
        'true_head': y_true_head,
        'pred_head': y_pred_head
    })
    
    comp_var = df_eval.groupby('comp').agg({
        'true_mod': 'std', 'pred_mod': 'std',
        'true_head': 'std', 'pred_head': 'std'
    }).fillna(0)
    
    var_spearman_mod, _ = spearmanr(comp_var['true_mod'], comp_var['pred_mod'])
    var_spearman_head, _ = spearmanr(comp_var['true_head'], comp_var['pred_head'])
    sub_var_spearman = (np.nan_to_num(var_spearman_mod) + np.nan_to_num(var_spearman_head)) / 2.0
    
    # 3. Auxiliary metric: RMSE
    rmse_mod = np.sqrt(mean_squared_error(y_true_mod, y_pred_mod))
    rmse_head = np.sqrt(mean_squared_error(y_true_head, y_pred_head))
    avg_rmse = (rmse_mod + rmse_head) / 2.0
    
    return {
        'Spearman_Mod': spearman_mod,
        'Spearman_Head': spearman_head,
        'Main_Spearman': main_spearman,
        'Var_Spearman_Mod': var_spearman_mod,
        'Var_Spearman_Head': var_spearman_head,
        'Sub_Var_Spearman': sub_var_spearman,
        'RMSE_Mod': rmse_mod,
        'RMSE_Head': rmse_head,
        'Avg_RMSE': avg_rmse
    }


# 6. Phương pháp 1: Vector tĩnh (Static Embeddings Cosine)


In [ ]:
class StaticEmbeddingBaseline:
    def __init__(self, embedding_dict=None):
        self.embedding_dict = embedding_dict if embedding_dict is not None else {}
        self.mod_mean = 2.5
        self.head_mean = 2.5

    def fit(self, train_df):
        self.mod_mean = train_df['Mod'].mean()
        self.head_mean = train_df['Head'].mean()

    def _cosine(self, v1, v2):
        norm1 = np.linalg.norm(v1)
        norm2 = np.linalg.norm(v2)
        if norm1 == 0 or norm2 == 0:
            return 0.5
        return float(np.dot(v1, v2) / (norm1 * norm2))

    def predict(self, df):
        preds_mod = []
        preds_head = []
        for _, row in df.iterrows():
            parts = str(row['Compound']).strip().split()
            if len(parts) >= 2:
                m_word, h_word = parts[0].lower(), parts[1].lower()
                c_phrase = "_".join(parts).lower()
                
                v_comp = self.embedding_dict.get(c_phrase, None)
                v_mod = self.embedding_dict.get(m_word, None)
                v_head = self.embedding_dict.get(h_word, None)
                
                if v_comp is not None and v_mod is not None:
                    sim_m = max(0.0, min(1.0, (self._cosine(v_comp, v_mod) + 1) / 2)) * 5.0
                else:
                    sim_m = self.mod_mean
                    
                if v_comp is not None and v_head is not None:
                    sim_h = max(0.0, min(1.0, (self._cosine(v_comp, v_head) + 1) / 2)) * 5.0
                else:
                    sim_h = self.head_mean
            else:
                sim_m, sim_h = self.mod_mean, self.head_mean
            preds_mod.append(sim_m)
            preds_head.append(sim_h)
        return np.array(preds_mod), np.array(preds_head)


# 7. Phương pháp 2: BERT không fine-tune (Feature Extraction)


In [ ]:
class UnsupervisedBERTBaseline:
    def __init__(self, model_name='bert-base-uncased', layer_idx=1):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        self.model.eval().to(device)
        self.layer_idx = layer_idx

    @torch.no_grad()
    def extract_features(self, contexts, compounds):
        preds_mod = []
        preds_head = []
        for ctx, comp in zip(contexts, compounds):
            parts = comp.strip().split()
            m_word, h_word = parts[0], parts[1] if len(parts) >= 2 else ""
            
            inputs = self.tokenizer(ctx, return_tensors='pt', truncation=True, max_length=128).to(device)
            outputs = self.model(**inputs)
            hidden = outputs.hidden_states[self.layer_idx][0]
            
            ctx_tokens = self.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
            ctx_text = " ".join(ctx_tokens).lower()
            
            m_emb = hidden[1:-1].mean(dim=0)
            h_emb = hidden[1:-1].mean(dim=0)
            comp_emb = hidden[1:-1].mean(dim=0)
            
            sim_m = float(torch.cosine_similarity(comp_emb.unsqueeze(0), m_emb.unsqueeze(0)))
            sim_h = float(torch.cosine_similarity(comp_emb.unsqueeze(0), h_emb.unsqueeze(0)))
            
            preds_mod.append(np.clip((sim_m + 1) / 2 * 5.0, 0, 5))
            preds_head.append(np.clip((sim_h + 1) / 2 * 5.0, 0, 5))
            
        return np.array(preds_mod), np.array(preds_head)


# 8. Phương pháp 3: BERT / DeBERTa có giám sát (Fine-tuning)


In [ ]:
class CompoundDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ctx = str(row['Context'])
        comp = str(row['Compound']).strip()
        parts = comp.split()
        m_word, h_word = parts[0], parts[1] if len(parts) >= 2 else ""
        
        marked_comp = f"<m>{m_word}</m> <h>{h_word}</h>"
        pattern = re.compile(re.escape(comp), re.IGNORECASE)
        marked_ctx = pattern.sub(marked_comp, ctx, count=1)
        
        inputs = self.tokenizer(
            marked_ctx,
            comp,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        if not self.is_test:
            item['targets'] = torch.tensor([row['Mod'], row['Head']], dtype=torch.float)
        return item

class DebertaRegressor(nn.Module):
    def __init__(self, model_name='microsoft/deberta-v3-base', num_added_tokens=4):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        if token_type_ids is not None:
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        else:
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs.last_hidden_state[:, 0, :]
        logits = self.head(cls_rep)
        return torch.sigmoid(logits) * 5.0


# 9. Huấn luyện và đánh giá


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)
        
        token_type_ids = batch.get('token_type_ids', None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
            
        preds = model(input_ids, attention_mask, token_type_ids)
        loss = criterion(preds, targets)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    all_preds = []
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch.get('token_type_ids', None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
            
        preds = model(input_ids, attention_mask, token_type_ids)
        all_preds.append(preds.cpu().numpy())
    return np.vstack(all_preds)

def run_fine_tuning(train_df, val_df, model_name='microsoft/deberta-v3-base', epochs=3, batch_size=16, lr=2e-5):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    special_tokens = {'additional_special_tokens': ['<m>', '</m>', '<h>', '</h>']}
    tokenizer.add_special_tokens(special_tokens)
    
    train_dataset = CompoundDataset(train_df, tokenizer)
    val_dataset = CompoundDataset(val_df, tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    model = DebertaRegressor(model_name=model_name).to(device)
    model.encoder.resize_token_embeddings(len(tokenizer))
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)
    criterion = nn.MSELoss()
    
    for epoch in range(epochs):
        train_loss = train_epoch(model, loader=train_loader, optimizer=optimizer, scheduler=scheduler, criterion=criterion)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {train_loss:.4f}")
        
    val_preds = evaluate_model(model, val_loader)
    metrics = compute_metrics(
        y_true_mod=val_df['Mod'].values,
        y_pred_mod=val_preds[:, 0],
        y_true_head=val_df['Head'].values,
        y_pred_head=val_preds[:, 1],
        compounds=val_df['Compound'].values
    )
    return model, metrics, val_preds


# 10. Phân tích lỗi (Error Analysis - Top 10 câu dự đoán sai)


In [ ]:
def analyze_top_errors(val_df, val_preds, n=10):
    analysis_df = val_df.copy().reset_index(drop=True)
    analysis_df['Pred_Mod'] = val_preds[:, 0]
    analysis_df['Pred_Head'] = val_preds[:, 1]
    
    analysis_df['Err_Mod'] = np.abs(analysis_df['Mod'] - analysis_df['Pred_Mod'])
    analysis_df['Err_Head'] = np.abs(analysis_df['Head'] - analysis_df['Pred_Head'])
    analysis_df['Total_Err'] = analysis_df['Err_Mod'] + analysis_df['Err_Head']
    
    top_errors = analysis_df.sort_values(by='Total_Err', ascending=False).head(n)
    
    print(f"=== TOP {n} CÂU CÓ SAI SỐ DỰ ĐOÁN LỚN NHẤT ===")
    for idx, row in top_errors.iterrows():
        print(f"\n--- Mẫu #{idx+1} [ContextID: {row.get('ContextID', 'N/A')}] ---")
        print(f"Compound: {row['Compound']}")
        print(f"Ngữ cảnh: {row['Context']}")
        print(f"Gold: Mod={row['Mod']:.2f} (std={row.get('ModStd', 0):.2f}) | Head={row['Head']:.2f} (std={row.get('HeadStd', 0):.2f})")
        print(f"Pred: Mod={row['Pred_Mod']:.2f} | Head={row['Pred_Head']:.2f} | Total Abs Err={row['Total_Err']:.2f}")
        
        # Phân tích nguyên nhân lỗi dựa trên đặc điểm ngôn ngữ học
        reasons = []
        if row.get('ModStd', 0) > 1.2 or row.get('HeadStd', 0) > 1.2:
            reasons.append("Bất đồng cao giữa người chấm (ModStd/HeadStd lớn)")
        if row['Mod'] < 1.0 and row['Head'] < 1.0:
            reasons.append("Thành ngữ hoàn toàn (Idiosyncratic/Idiomatic shift)")
        elif row['Mod'] > 4.0 and row['Head'] > 4.0:
            reasons.append("Nghĩa đen / Hợp thành hoàn toàn nhưng mô hình đoán lệch")
        else:
            reasons.append("Nghĩa bóng từng phần / Văn cảnh lịch sử hoặc đa nghĩa")
            
        print(f"Nguyên nhân tiềm ẩn: {'; '.join(reasons)}")
        
    return top_errors
